In [19]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.colors import HexColor
from reportlab.graphics import renderPDF
from svglib.svglib import svg2rlg
from reportlab.lib.utils import simpleSplit


def draw_justified_text(c, text, x, y, max_width, font, font_size, line_height=12):
    c.setFont(font, font_size)

    lines = simpleSplit(text, font, font_size, max_width)

    for i, line in enumerate(lines):

        # dernière ligne → alignée à gauche
        if i == len(lines) - 1:
            c.drawString(x, y, line)

        else:
            words = line.split(" ")
            if len(words) == 1:
                c.drawString(x, y, line)
            else:
                total_words_width = sum(c.stringWidth(w, font, font_size) for w in words)
                space_width = c.stringWidth(" ", font, font_size)

                total_space = max_width - total_words_width
                space_between = total_space / (len(words) - 1)

                current_x = x

                for w in words:
                    c.drawString(current_x, y, w)
                    current_x += c.stringWidth(w, font, font_size) + space_between

        y -= line_height

    return y


def draw_colored_line(c, debut_x, y, longueur, couleur):
    c.saveState()
    c.translate(debut_x, y)
    c.setFillColor(couleur)
    c.rect(0, 0, longueur, 1, fill=1, stroke=0)
    c.restoreState()

def draw_bandeau(c, debut_x, debut_y, longueur, couleur, titre):
    c.saveState()
    c.translate(debut_x - 5, debut_y - 5)
    c.setFillColor(couleur)
    c.rect(0, 0, longueur, 20, fill=1, stroke=0)

    # c.translate(debut_x + longueur - 20, debut_y - 5)
    # c.translate(debut_x, debut_y - 5)
    c.translate(longueur - 23, -3)
    c.rotate(49)
    
    c.setFillColor(white)
    c.rect(0, 0, 30, 4, fill=1, stroke=0)

    c.translate(10,-10)
    c.rect(0, 0, 30, 4, fill=1, stroke=0)

    c.restoreState()
    c.setFont(police, size_big_text)
    c.drawString(x, y, f"{titre}")
    

# ========================
# INITIALISATION
# ========================
c = canvas.Canvas("cv_samuel_ramirez.pdf", pagesize=A4)
width, height = A4
police = "Helvetica-Bold"
police_2 = "Helvetica"
size_text = 9
size_big_text = 12

# Couleurs
blue = HexColor("#284E73")
light_blue = HexColor("#ADD1EA")
# couleur_colonne = HexColor("#317A34")
couleur_colonne = blue
light_grey = HexColor("#F2F2F2")
dark = HexColor("#333333")
white = HexColor("#FFFFFF")

# Largeur colonne gauche
left_w = 135
x = left_w/2
marge_colonne = 12
bullet = "•"

# ========================
# FOND COLONNE GAUCHE
# ========================
c.setFillColor(couleur_colonne)
c.rect(0, 0, left_w, height, fill=1, stroke=0)

# ========================
# PHOTO
# ========================
y = height
largeur_photo = 86
hauteur_photo = 115
photo_x = x - largeur_photo/2
y = y - (hauteur_photo + 20)
photo_y = y
c.drawImage(r"D:\CV2026\photo_cv.png", photo_x, photo_y, width=largeur_photo, height=hauteur_photo)

# ========================
# NOM ET DATE DE NAISSANCE
# ========================
c.setFillColor(white)
c.setFont(police, size_big_text)
y -= 25
c.drawCentredString(x, y, "Samuel RAMIREZ")
c.setFont(police_2, size_text)
y -= 15
c.drawCentredString(x, y, "5 Février 1995")

# Ligne séparatrice
c.saveState()
longueur_trait = left_w - 20
y -= 12
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15

# ========================
# CONTACT
# ========================
contact = [
    ("phone.svg", "+33 6 81 44 65 34", None),
    ("envelope.svg", "samuel.reinaldo.ramirez\n@gmail.com", None),
    ("home.svg", "Arcueil 94110, France", None),
    # ("car.svg", "Permis B", None),
    ("linkedin.svg", "LinkedIn", "https://www.linkedin.com/in/samuel-ramirez-53064718b/")
]


c.setFont(police_2, size_text)
icon_size = 12
line_height = 12

for svg_file, text, link in contact:
    lines = text.split("\n")
    n_lines = len(lines)

    # Charger et recolorer SVG
    drawing = svg2rlg(svg_file)
    def recolor(shape):
        if hasattr(shape, 'fillColor') and shape.fillColor is not None:
            shape.fillColor = white
        if hasattr(shape, 'strokeColor') and shape.strokeColor is not None:
            shape.strokeColor = white
        if hasattr(shape, 'contents'):
            for s in shape.contents:
                recolor(s)
    recolor(drawing)

    # Redimensionner
    scale = icon_size / max(drawing.width, drawing.height)
    drawing.width *= scale
    drawing.height *= scale

    # Dessiner l'icône alignée à la première ligne
    icon_y = y - (line_height * (n_lines - 1))/2 - icon_size/2 + 4
    renderPDF.draw(drawing, c, marge_colonne - 2, icon_y)
    

    # Dessiner texte
    for line in lines:
        c.drawString(marge_colonne + icon_size + 3, y, line)
        y -= line_height

    # longueur_hypertext = marge_colonne + icon_size + 40
    longueur_hypertext = left_w - marge_colonne
    debut_hypertext_x = marge_colonne - 2
    # Lien
    if link:
        c.linkURL(link, (debut_hypertext_x, y + 5, longueur_hypertext, y + line_height*n_lines+10), relative=0)
        # c.linkURL(link, (20, y-2, left_w-50, y + line_height*n_lines), relative=0)

    y -= 4

# ========================
# LANGUES
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Langues")
c.setFont(police_2, size_text)
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y-=15

# langs = [
#     "Français : langue maternelle",
#     "Anglais : 850 au TOEIC",
#     "Espagnol : langue maternelle"
# ]

langs = [
    "Français : C2",
    "Anglais : C1",
    "Espagnol : C2",
    "Hébreu : B1",
    # "Chinois : A1"
]

for l in langs:
    c.drawString(marge_colonne, y, l)
    y -= 12

# ========================
# CERTIFICATIONS
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Certifications")
c.setFont(police_2, size_text)
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y-=15

certs = [
    "Coursera : Machine Learning \nStanford Uni. par Andrew Ng",
    "OCA Java SE 8",
    "OCP Java SE 8",
    "AWS Certified Cloud \nPractitioner",
    "Angular Certification level 1"
]

for ccc in certs:
    c.drawString(marge_colonne - 5, y, bullet)
    for line in ccc.split("\n"):
        c.drawString(marge_colonne, y, line)
        y -= 8
    y -= 4



# ========================
# COMPETENCES
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16

c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Compétences")
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15
c.setFont(police_2, size_text)
# skills = [
#     "Versioning (Git), Agilité (Scrum)",
#     "Bases de données SQL, AWS, Xano",
#     "Matlab, POO, Programmation fonctionnelle",
#     "Métaheuristiques, ETL : Talend, Databricks, PySpark",
#     "Développement applicatif : ",
#     "Front : Javascript, HTML, CSS, Angular, React",
#     "Back : Java 8, Python, REST",
#     "Mobile : Android, Flutter"
# ]

# # Versioning (Git), Agilité (scrum), Algorithmie, Bases de données SQL, AWS, Xano,
# # Utilisation du terminal, Matlab, Programmation orientée objet, Programmation
# # fonctionnelle, Métaheuristiques, ETL : Talend, Databricks, PySpark
# # Développement applicatif :
# # Front : Javascript, HTML, css, Angular, React
# # Back : Java 8 et technologies associés, Python, REST
# # Mobile : Android, Flutter


# max_width = left_w - marge_colonne - 10

# for s in skills:
#     lines = simpleSplit(s, police_2, size_text, max_width)

#     for line in lines:
#         c.drawString(marge_colonne, y, line)
#         y -= 10

#     y -= 2

skills = [
    "Machine Learning\nSupervisé, non supervisé,\nfeature engineering",
    
    "Clustering & anomalies\nRéduction de dimension\nSéries temporelles",
    
    "Deep Learning\nKeras, CNN, OpenCV",
    
    "Data Engineering\nSQL, Talend, PySpark",
    
    "MLOps\nDocker, MLflow, DVC\nAirflow, BentoML",
    
    "Monitoring\nPrometheus, Grafana",
    
    "APIs & Serving\nFastAPI, REST, AWS",
    
    "Développement\nPython, Java, React, Git\nLinux / Bash, Android, Flutter",
    
    "Méthodes\nAgile (Scrum)"
]

for s in skills:

    c.drawString(marge_colonne - 5, y, bullet)

    first = True
    for line in s.split("\n"):

        if first:
            c.setFont(police, size_text)  # titre en gras
            c.drawString(marge_colonne, y, line)

            text_width = c.stringWidth(line, police, size_text)
            c.line(marge_colonne, y - 1, marge_colonne + text_width, y - 1)

            first = False
        else:
            c.setFont(police_2, size_text)
            c.drawString(marge_colonne, y, line)

        y -= 8

    y -= 4


# ========================
# CENTRES D'INTERET
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Centres d'intérêt")
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15
c.setFont(police_2, size_text)
interets = [
    "Judo  " + bullet + "  Futsal  " + bullet + "  Cyclisme",
    "Échecs  " + bullet + "  E-Sport  " + bullet + "  Trading", 
    "Développement mobile  "
]
for i in interets:
    c.drawString(marge_colonne, y, i)
    y -= 12
# ========================
# COLONNE DROITE
# ========================
marge_colonne_droite = 10
x_title = (width + left_w) / 2
x = left_w + marge_colonne_droite
y = height - 30
c.setFillColor(dark)

# Titre
c.setFont(police, 18)
c.drawCentredString(x_title, y, "Ingénieur en Informatique - Data Scientist")

# ========================
# PROFIL
# ========================
y -= 40
c.setFont(police, size_big_text)
c.drawString(x, y, "Profil")
draw_bandeau(c, x, y, 440, light_blue, "Profil")
y -= 20

profil_text = (
"Ingénieur en informatique fasciné par l’intelligence artificielle et la "
"modélisation prédictive. Après cinq ans comme développeur web, j’ai exploré "
"l’entrepreneuriat en co-créant l’application mobile Yummap, avant de me "
"spécialiser en Data Science. Actuellement Data Scientist en alternance chez "
"AXA, je conçois des modèles data tout en développant des projets personnels "
"en machine learning, notamment un modèle de prédiction de l’issue d’une "
"partie de League of Legends à partir de la draft."
)

max_width = width - x - 40

y = draw_justified_text(
    c,
    profil_text,
    x,
    y,
    max_width,
    police_2,
    size_text,
    11
)

# # ========================
# # EXPERIENCES PROFESSIONNELLES
# # ========================
# y -= 20
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Expériences professionnelles")
# y -= 20

# date_color = blue

# exp = [

# ("Juin 2025 – Juin 2026", "AXA – Data Scientist (alternance)", [
# "Développement et optimisation d’un algorithme de name matching pour la détection""de \npersonnes sous sanction dans la base clients.",
# "Réduction du ratio de faux positifs de 4 alertes pour 1 réelle à 1,7 ", "grâce à l’exploration de combinaisons de distances de chaînes.",
# "Optimisation du temps de traitement de plusieurs heures à ~50 minutes ", "via des méthodes de clustering (médoïdes).",
# "Étude d’un système de classification automatique des alertes liées aux ", "Personnes Politiquement Exposées (PPE)."
# ]),

# ("Fév 2024 – Jan 2025", "Yummap – Co-fondateur", [
# "Conception et développement d’une application mobile de découverte de restaurants.",
# "Développement Flutter et application d’administration (WeWeb puis Flutter).",
# "Architecture backend : Xano, stockage AWS S3.",
# "Marketing et acquisition : création de contenu, campagnes emailing.",
# "Prospection commerciale et rendez-vous clients."
# ]),

# ("Fév 2018 – Déc 2023", "Oxyl – Consultant développeur WEB", [
# "Lafarge Holcim : maintenance et évolution de l’ERP Quartz (Java).",
# "La Réunion Aérienne : évolution d’applications Java / JavaScript.",
# "Wedia : développement d’une solution de workflows personnalisés (cycle en V).",
# "Projets internes : My Neuro Factory (React, NodeJS, SQL, Scrum).",
# "Développement d’outils internes (React, NodeJS, Puppeteer)."
# ]),

# (["00 - 00", "AA - ZZZ", ["parler de vinted, du travail à la ferme, des cours particuliers ?"]])

# ]

# for date, company, bullets in exp:
#     y -= 10
#     # date
#     c.setFillColor(date_color)
#     c.setFont(police_2 + "-Oblique", size_text)
#     c.drawString(x, y, date)

#     # entreprise
#     c.setFillColor(dark)
#     c.setFont(police, size_text)
#     c.drawString(x, y + 10, company)

#     y -= 12

#     # bullets
#     c.setFont(police_2, size_text)
#     for b in bullets:
#         c.drawString(x + 10, y, "• " + b)
#         y -= 11

#     y -= 6

# ========================
# EXPERIENCES PROFESSIONNELLES
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Expériences professionnelles")
y -= 20

date_color = blue

exp = [

("Juin 2025 – Juin 2026", "AXA – Data Scientist (alternance)", [
"Développement et optimisation d’un algorithme de name matching pour la détection de personnes \nsous sanction dans la base clients.",
"Réduction du ratio de faux positifs de 4 alertes pour 1 réelle à 1,7 grâce à l’exploration de \ncombinaisons de distances de chaînes.",
"Optimisation du temps de traitement de plusieurs heures à ~50 minutes via des méthodes \nde clustering (médoïdes).",
"Étude d’un système de classification automatique des alertes liées aux \nPersonnes Politiquement Exposées (PPE)."
]),

("Fév 2024 – Jan 2025", "Yummap – Co-fondateur", [
"Conception et développement d’une application mobile de découverte de restaurants.",
"Développement Flutter et application d’administration (WeWeb puis Flutter).",
"Architecture backend : Xano, stockage AWS S3.",
"Marketing et acquisition : création de contenu, campagnes emailing.",
"Prospection commerciale et rendez-vous clients."
]),

("Fév 2018 – Déc 2023", "Oxyl – Consultant développeur WEB", [
"Lafarge Holcim : maintenance et évolution de l’ERP Quartz (Java).",
"La Réunion Aérienne : évolution d’applications Java / JavaScript.",
"Wedia : développement d’une solution de workflows personnalisés (cycle en V).",
"Projets internes : My Neuro Factory (React, NodeJS, SQL, Scrum).",
"Développement d’outils internes (React, NodeJS, Puppeteer)."
]),

# ("00 - 00", "AA - ZZZ", [
# "Parler de Vinted, du travail à la ferme, des cours particuliers ?"
# ])

]

for date, company, bullets in exp:
    y -= 10
    # date
    c.setFillColor(date_color)
    c.setFont(police_2 + "-Oblique", size_text)
    c.drawString(x, y, date)

    # entreprise
    c.setFillColor(dark)
    c.setFont(police, size_text)
    c.drawString(x, y + 10, company)

    y -= 12

    # bullets avec gestion du \n
    c.setFont(police_2, size_text)
    for b in bullets:
        lines = b.split("\n")
        for i, line in enumerate(lines):
            if i == 0:
                # première ligne → puce
                c.drawString(x + 10, y, "• " + line)
            else:
                # lignes suivantes → pas de puce, juste indentées
                c.drawString(x + 20, y, line)
            y -= 11

    y -= 6


# # ========================
# # EXPERIENCES PROFESSIONNELLES
# # ========================
# y -= 20
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Expériences professionnelles")
# y -= 20
# c.setFont(police_2, size_text)
# exp = [
#     "Juin 2025 – Juin 2026 : Data Scientist (contrat de professionnalisation) chez Axa",
#     "• Amélioration continue de l’algorithme de name matching VS Netreveal",
#     "• Mise à jour des SIREN, classification des alertes PPE",
#     "Fév 2024 – Jan 2025 : Co-créateur de Yummap",
#     "• Étude du marché, développement de l’application Flutter",
#     "• Application admin (Weweb puis Flutter) + pros. Stockage : Xano, AWS S3",
#     "• Marketing : création de contenu vidéo, campagnes emailing",
#     "• Vente : contacts téléphoniques, rendez-vous physiques",
#     "Sept 2018 – Déc 2023 : Consultant ingénieur chez Oxyl",
#     "• Lafarge Holcim : maintenance Quartz (Java)",
#     "• La Réunion Aérienne : évolution projets Java/JS",
#     "• Wedia : solution workflows personnalisés (cycle en V)",
#     "• Projets internes : My Neuro Factory (React, NodeJS, SQL, Scrum)",
#     "• Autocom (React, NodeJS, Pupeteer)",
#     "• Capico (AngularJS, Scrum)"
# ]
# for e in exp:
#     c.drawString(x, y, e)
#     y -= 12

# ========================
# PROJETS PERSONNELS
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Projets personnels")
y -= 20
c.setFont(police_2, size_text)

projects = [
    ("Assistant Clash Royal", [
        "but : donner les information de spectateur et créer un bot joueur",
        "Python, PyCharm, OpenCV, pyautogui"
    ]),
    ("Plugin Vinted (Android)", [
        "but : republier des annonces en un clic, générer une publication en fonction des photos",
        "Kotlin, Google MLkit, OpenCV, AccessibilityService"
    ]),
    ("Draft Analyzer (projet d'école)", [
        "Partie DPM : conception pour aider la communauté LoL et bookmakers",
        "Collecte de données (API + scraping) : compositions d'équipes, données sur champions et invocateurs",
        "Pipeline ML : XGBoost, fonction beta",
        # "Partie MLOps : pas encore commencée"
    ])
]

for title, details in projects:
    # Puce + titre du projet
    c.drawString(x - 5, y, bullet)
    c.setFont(police, size_text)
    c.drawString(x, y, title)
    text_width = c.stringWidth(title, police, size_text)
    # c.line(x, y - 1, x + text_width, y - 1)  # souligne le titre
    y -= 10

    # Détails du projet, retours à la ligne
    c.setFont(police_2, size_text)
    for d in details:
        for line in d.split("\n"):
            c.drawString(x + 10, y, line)
            y -= 10
    y -= 6

# # ========================
# # PROJETS PERSONNELS
# # ========================
# y -= 20
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Projets personnels")
# # c.drawString(x, y, "Projets personnels")
# y -= 20
# c.setFont(police_2, size_text)
# projects = [
#     "Assistant Clash Royal : Python, PyCharm, openCV, pyautogui",
#     "Plugin Vinted (Android) : Kotlin, Google MLkit, OpenCV, AccessibilityService"
# ]
# for p in projects:
#     c.drawString(x, y, p)
#     y -= 12

# # ========================
# # FORMATION
# # ========================
# y -= 40
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Formation")
# # c.drawString(x, y, "Formation")
# y -= 20
# c.setFont(police_2, size_text)

# formation = [
#     "Juin 2025 - Juin 2026 : Formation Machine Learning Engineer chez Datascientest (Axa)",
#     "2015-2018 : ENSSAT Lannion - diplôme d’ingénieur en informatique",
#     "2018 : Erasmus Universidad Politécnica, Madrid (métaheuristiques, biologie programmable, modèles de raisonnement)",
#     "2013-2015 : CPGE PCSI-PSI au Lycée Michelet de Vanves"
# ]

# for f in formation:
#     c.drawString(x, y, f)
#     y -= 14


# ========================
# FORMATION
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Formation")
y -= 20

date_color = blue

formation = [
    ("Juin 2025 - Juin 2026", "Datascientest / Axa", "Formation Machine Learning Engineer – alternance Data Science"),
    ("Sept 2015 - Juin 2018", "ENSSAT Lannion", "Diplôme d’ingénieur en informatique"),
    ("Sept 2017 - Fév 2018", "Universidad Politécnica, Madrid", "Erasmus – Métaheuristiques, biologie programmable, modèles de raisonnement"),
    ("Sept 2013 - Juin 2015", "Lycée Michelet de Vanves", "CPGE PCSI-PSI")
]

for date, school, detail in formation:
    y-=10
    # Date
    c.setFillColor(date_color)
    c.setFont(police_2 + "-Oblique", size_text)
    c.drawString(x, y, date)

    # École / établissement
    c.setFillColor(dark)
    c.setFont(police, size_text)
    c.drawString(x, y + 10, school)

    y -= 12

    # Détails / diplôme / spécialité
    c.setFont(police_2, size_text)
    c.drawString(x, y, detail)
    y -= 15



# ========================
# ENREGISTREMENT PDF
# ========================
c.save()

In [21]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.colors import HexColor
from reportlab.graphics import renderPDF
from svglib.svglib import svg2rlg
from reportlab.lib.utils import simpleSplit


def draw_justified_text(c, text, x, y, max_width, font, font_size, line_height=12):
    c.setFont(font, font_size)

    lines = simpleSplit(text, font, font_size, max_width)

    for i, line in enumerate(lines):

        # dernière ligne → alignée à gauche
        if i == len(lines) - 1:
            c.drawString(x, y, line)

        else:
            words = line.split(" ")
            if len(words) == 1:
                c.drawString(x, y, line)
            else:
                total_words_width = sum(c.stringWidth(w, font, font_size) for w in words)
                space_width = c.stringWidth(" ", font, font_size)

                total_space = max_width - total_words_width
                space_between = total_space / (len(words) - 1)

                current_x = x

                for w in words:
                    c.drawString(current_x, y, w)
                    current_x += c.stringWidth(w, font, font_size) + space_between

        y -= line_height

    return y


def draw_colored_line(c, debut_x, y, longueur, couleur):
    c.saveState()
    c.translate(debut_x, y)
    c.setFillColor(couleur)
    c.rect(0, 0, longueur, 1, fill=1, stroke=0)
    c.restoreState()

def draw_bandeau(c, debut_x, debut_y, longueur, couleur, titre):
    c.saveState()
    c.translate(debut_x - 5, debut_y - 5)
    c.setFillColor(couleur)
    c.rect(0, 0, longueur, 20, fill=1, stroke=0)

    # c.translate(debut_x + longueur - 20, debut_y - 5)
    # c.translate(debut_x, debut_y - 5)
    c.translate(longueur - 23, -3)
    c.rotate(49)
    
    c.setFillColor(white)
    c.rect(0, 0, 30, 4, fill=1, stroke=0)

    c.translate(10,-10)
    c.rect(0, 0, 30, 4, fill=1, stroke=0)

    c.restoreState()
    c.setFont(police, size_big_text)
    c.drawString(x, y, f"{titre}")
    

# ========================
# INITIALISATION
# ========================
c = canvas.Canvas("cv_samuel_ramirez.pdf", pagesize=A4)
width, height = A4
police = "Helvetica-Bold"
police_2 = "Helvetica"
size_text = 9
size_big_text = 12

# Couleurs
blue = HexColor("#284E73")
light_blue = HexColor("#ADD1EA")
# couleur_colonne = HexColor("#317A34")
couleur_colonne = blue
light_grey = HexColor("#F2F2F2")
dark = HexColor("#333333")
white = HexColor("#FFFFFF")

# Largeur colonne gauche
left_w = 135
x = left_w/2
marge_colonne = 12
bullet = "•"

# ========================
# FOND COLONNE GAUCHE
# ========================
c.setFillColor(couleur_colonne)
c.rect(0, 0, left_w, height, fill=1, stroke=0)

# ========================
# PHOTO
# ========================
y = height
largeur_photo = 86
hauteur_photo = 115
photo_x = x - largeur_photo/2
y = y - (hauteur_photo + 20)
photo_y = y
c.drawImage(r"D:\CV2026\photo_cv.png", photo_x, photo_y, width=largeur_photo, height=hauteur_photo)

# ========================
# NOM ET DATE DE NAISSANCE
# ========================
c.setFillColor(white)
c.setFont(police, size_big_text)
y -= 25
c.drawCentredString(x, y, "Samuel RAMIREZ")
c.setFont(police_2, size_text)
y -= 15
c.drawCentredString(x, y, "5 Février 1995")

# Ligne séparatrice
c.saveState()
longueur_trait = left_w - 20
y -= 12
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15

# ========================
# CONTACT
# ========================
contact = [
    ("phone.svg", "+33 6 81 44 65 34", None),
    ("envelope.svg", "samuel.reinaldo.ramirez\n@gmail.com", None),
    ("home.svg", "Arcueil 94110, France", None),
    # ("car.svg", "Permis B", None),
    ("linkedin.svg", "LinkedIn", "https://www.linkedin.com/in/samuel-ramirez-53064718b/")
]


c.setFont(police_2, size_text)
icon_size = 12
line_height = 12

for svg_file, text, link in contact:
    lines = text.split("\n")
    n_lines = len(lines)

    # Charger et recolorer SVG
    drawing = svg2rlg(svg_file)
    def recolor(shape):
        if hasattr(shape, 'fillColor') and shape.fillColor is not None:
            shape.fillColor = white
        if hasattr(shape, 'strokeColor') and shape.strokeColor is not None:
            shape.strokeColor = white
        if hasattr(shape, 'contents'):
            for s in shape.contents:
                recolor(s)
    recolor(drawing)

    # Redimensionner
    scale = icon_size / max(drawing.width, drawing.height)
    drawing.width *= scale
    drawing.height *= scale

    # Dessiner l'icône alignée à la première ligne
    icon_y = y - (line_height * (n_lines - 1))/2 - icon_size/2 + 4
    renderPDF.draw(drawing, c, marge_colonne - 2, icon_y)
    

    # Dessiner texte
    for line in lines:
        c.drawString(marge_colonne + icon_size + 3, y, line)
        y -= line_height

    # longueur_hypertext = marge_colonne + icon_size + 40
    longueur_hypertext = left_w - marge_colonne
    debut_hypertext_x = marge_colonne - 2
    # Lien
    if link:
        c.linkURL(link, (debut_hypertext_x, y + 5, longueur_hypertext, y + line_height*n_lines+10), relative=0)
        # c.linkURL(link, (20, y-2, left_w-50, y + line_height*n_lines), relative=0)

    y -= 4

# ========================
# LANGUES
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Langues")
c.setFont(police_2, size_text)
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y-=15

# langs = [
#     "Français : langue maternelle",
#     "Anglais : 850 au TOEIC",
#     "Espagnol : langue maternelle"
# ]

langs = [
    "Français : C2",
    "Anglais : C1",
    "Espagnol : C2",
    "Hébreu : B1",
    # "Chinois : A1"
]

for l in langs:
    c.drawString(marge_colonne, y, l)
    y -= 12

# ========================
# CERTIFICATIONS
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Certifications")
c.setFont(police_2, size_text)
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y-=15

certs = [
    "Coursera : Machine Learning \nStanford Uni. par Andrew Ng",
    "OCA Java SE 8",
    "OCP Java SE 8",
    "AWS Certified Cloud \nPractitioner",
    "Angular Certification level 1"
]

for ccc in certs:
    c.drawString(marge_colonne - 5, y, bullet)
    for line in ccc.split("\n"):
        c.drawString(marge_colonne, y, line)
        y -= 8
    y -= 4



# ========================
# COMPETENCES
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16

c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Compétences")
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15
c.setFont(police_2, size_text)
# skills = [
#     "Versioning (Git), Agilité (Scrum)",
#     "Bases de données SQL, AWS, Xano",
#     "Matlab, POO, Programmation fonctionnelle",
#     "Métaheuristiques, ETL : Talend, Databricks, PySpark",
#     "Développement applicatif : ",
#     "Front : Javascript, HTML, CSS, Angular, React",
#     "Back : Java 8, Python, REST",
#     "Mobile : Android, Flutter"
# ]

# # Versioning (Git), Agilité (scrum), Algorithmie, Bases de données SQL, AWS, Xano,
# # Utilisation du terminal, Matlab, Programmation orientée objet, Programmation
# # fonctionnelle, Métaheuristiques, ETL : Talend, Databricks, PySpark
# # Développement applicatif :
# # Front : Javascript, HTML, css, Angular, React
# # Back : Java 8 et technologies associés, Python, REST
# # Mobile : Android, Flutter


# max_width = left_w - marge_colonne - 10

# for s in skills:
#     lines = simpleSplit(s, police_2, size_text, max_width)

#     for line in lines:
#         c.drawString(marge_colonne, y, line)
#         y -= 10

#     y -= 2

skills = [
    "Machine Learning\nSupervisé, non supervisé,\nfeature engineering",
    
    "Clustering & anomalies\nRéduction de dimension\nSéries temporelles",
    
    "Deep Learning\nKeras, CNN, OpenCV",
    
    "Data Engineering\nSQL, Talend, PySpark",
    
    "MLOps\nDocker, MLflow, DVC\nAirflow, BentoML",
    
    "Monitoring\nPrometheus, Grafana",
    
    "APIs & Serving\nFastAPI, REST, AWS",
    
    "Développement\nPython, Java, React, Git\nLinux / Bash, Android, Flutter",
    
    "Méthodes\nAgile (Scrum)"
]

for s in skills:

    c.drawString(marge_colonne - 5, y, bullet)

    first = True
    for line in s.split("\n"):

        if first:
            c.setFont(police, size_text)  # titre en gras
            c.drawString(marge_colonne, y, line)

            text_width = c.stringWidth(line, police, size_text)
            c.line(marge_colonne, y - 1, marge_colonne + text_width, y - 1)

            first = False
        else:
            c.setFont(police_2, size_text)
            c.drawString(marge_colonne, y, line)

        y -= 8

    y -= 4


# ========================
# CENTRES D'INTERET
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Centres d'intérêt")
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15
c.setFont(police_2, size_text)
interets = [
    "Judo  " + bullet + "  Futsal  " + bullet + "  Cyclisme",
    "Échecs  " + bullet + "  E-Sport  " + bullet + "  Trading", 
    "Développement mobile  "
]
for i in interets:
    c.drawString(marge_colonne, y, i)
    y -= 12
# ========================
# COLONNE DROITE
# ========================
marge_colonne_droite = 10
x_title = (width + left_w) / 2
x = left_w + marge_colonne_droite
y = height - 30
c.setFillColor(dark)

# Titre
c.setFont(police, 18)
c.drawCentredString(x_title, y, "Ingénieur en Informatique - Data Scientist")

# ========================
# PROFIL
# ========================
y -= 40
c.setFont(police, size_big_text)
c.drawString(x, y, "Profil")
draw_bandeau(c, x, y, 440, light_blue, "Profil")
y -= 20

profil_text = (
"Ingénieur en informatique fasciné par l’intelligence artificielle et la "
"modélisation prédictive. Après cinq ans comme développeur web, j’ai exploré "
"l’entrepreneuriat en co-créant l’application mobile Yummap, avant de me "
"spécialiser en Data Science. Actuellement Data Scientist en alternance chez "
"AXA, je conçois des modèles data tout en développant des projets personnels "
"en machine learning, notamment un modèle de prédiction de l’issue d’une "
"partie de League of Legends à partir de la draft."
)

max_width = width - x - 40

y = draw_justified_text(
    c,
    profil_text,
    x,
    y,
    max_width,
    police_2,
    size_text,
    11
)


# # ========================
# # EXPERIENCES PROFESSIONNELLES
# # ========================
# y -= 20
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Expériences professionnelles")
# y -= 20

# date_color = blue

# exp = [

# ("Juin 2025 – Juin 2026", "AXA – Data Scientist (alternance)", [
# "Développement et optimisation d’un algorithme de name matching pour la détection de personnes \nsous sanction dans la base clients.",
# "Réduction du ratio de faux positifs de 4 alertes pour 1 réelle à 1,7 grâce à l’exploration de \ncombinaisons de distances de chaînes.",
# "Optimisation du temps de traitement de plusieurs heures à ~50 minutes via des méthodes \nde clustering (médoïdes).",
# "Étude d’un système de classification automatique des alertes liées aux \nPersonnes Politiquement Exposées (PPE)."
# ]),

# ("Fév 2024 – Jan 2025", "Yummap – Co-fondateur", [
# "Conception et développement d’une application mobile de découverte de restaurants.",
# "Développement Flutter et application d’administration (WeWeb puis Flutter).",
# "Architecture backend : Xano, stockage AWS S3.",
# "Marketing et acquisition : création de contenu, campagnes emailing.",
# "Prospection commerciale et rendez-vous clients."
# ]),

# ("Fév 2018 – Déc 2023", "Oxyl – Consultant développeur WEB", [
# "• Août 2022 - Oct 2023 : Développeur JS (React / Node), Arcueil\n"
# "Développement du projet Autocom pour automatiser le recrutement et la partie commerciale d’Oxyl. Scraping de sites, monitoring des bots, maintenance et évolution avec une équipe agile de 5 développeurs.",
# "• Oct 2021 - Juil 2022 : Développeur Java chez Lafarge Holcim, Clamart\n"
# "Maintenance et évolution de Quartz (ERP) pour la filière granulats. Nouvelle API pour stockage PDF, déploiements via GitLab.",
# "• Mai 2018 - Oct 2021 : Développeur web Capico, Arcueil\n"
# "Maintenance, évolution et tests unitaires du projet Java/Angular Capico. Suivi pédagogique et fonctionnel des cours et classes.",
# "• Sept 2019 - Juil 2022 : Développeur La Réunion Aérienne, Paris\n"
# "Maintenance et évolution des projets Java/JS NSI et Webreport en coopération avec la MOA.",
# "• Avr 2019 - Sept 2019 : Développeur / Scrum Master MNF (My Neuro Factory), Arcueil\n"
# "Application React/Java pour référencer les collaborateurs et faciliter la montée en compétences et certifications.",
# "• Déc 2018 - Avr 2019 : Développeur web consultant Wedia, Paris\n"
# "Développement d’une solution personnalisable de workflows (cycle en V), gestion des permissions selon type d’utilisateur."
# ])
# ]

# for date, company, bullets in exp:
#     y -= 10
#     # date
#     c.setFillColor(date_color)
#     c.setFont(police_2 + "-Oblique", size_text)
#     c.drawString(x, y, date)

#     # entreprise
#     c.setFillColor(dark)
#     c.setFont(police, size_text)
#     c.drawString(x, y + 10, company)

#     y -= 12

#     # bullets avec gestion du \n
#     c.setFont(police_2, size_text)
#     for b in bullets:
#         lines = b.split("\n")
#         for i, line in enumerate(lines):
#             if i == 0 and not line.startswith("Développement") and not line.startswith("Maintenance"):
#                 # première ligne → puce si ce n’est pas un retour à la ligne forcé
#                 c.drawString(x + 10, y, "• " + line)
#             else:
#                 # lignes suivantes → pas de puce, juste indentées
#                 c.drawString(x + 20, y, line)
#             y -= 11

#     y -= 6




# ========================
# EXPERIENCES PROFESSIONNELLES
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Expériences professionnelles")
y -= 20

date_color = blue

exp = [

("Juin 2025 – Juin 2026", "AXA – Data Scientist (alternance)", [
"Développement et optimisation d’un algorithme de name matching pour la détection de personnes \nsous sanction dans la base clients.",
"Réduction du ratio de faux positifs de 4 alertes pour 1 réelle à 1,7 grâce à l’exploration de \ncombinaisons de distances de chaînes.",
"Optimisation du temps de traitement de plusieurs heures à ~50 minutes via des méthodes \nde clustering (médoïdes).",
"Étude d’un système de classification automatique des alertes liées aux \nPersonnes Politiquement Exposées (PPE)."
]),

("Fév 2024 – Jan 2025", "Yummap – Co-fondateur", [
"Conception et développement d’une application mobile de découverte de restaurants.",
"Développement Flutter et application d’administration (WeWeb puis Flutter).",
"Architecture backend : Xano, stockage AWS S3.",
"Marketing et acquisition : création de contenu, campagnes emailing.",
"Prospection commerciale et rendez-vous clients."
]),

("Fév 2018 – Déc 2023", "Oxyl – Consultant développeur WEB", [
"Lafarge Holcim : maintenance et évolution de l’ERP Quartz (Java).",
"La Réunion Aérienne : évolution d’applications Java / JavaScript.",
"Wedia : développement d’une solution de workflows personnalisés (cycle en V).",
"Projets internes : My Neuro Factory (React, NodeJS, SQL, Scrum).",
"Développement d’outils internes (React, NodeJS, Puppeteer)."
]),

# ("00 - 00", "AA - ZZZ", [
# "Parler de Vinted, du travail à la ferme, des cours particuliers ?"
# ])

]

for date, company, bullets in exp:
    y -= 10

    # entreprise
    c.setFillColor(dark)
    c.setFont(police, size_text)
    c.drawString(x, y + 10, company)

    # date
    c.setFillColor(date_color)
    c.setFont(police_2 + "-Oblique", size_text)
    c.drawString(x + company_width, y + 10, date)

    

    y -= 12

    # bullets avec gestion du \n
    c.setFont(police_2, size_text)
    for b in bullets:
        lines = b.split("\n")
        for i, line in enumerate(lines):
            if i == 0:
                # première ligne → puce
                c.drawString(x + 10, y, "• " + line)
            else:
                # lignes suivantes → pas de puce, juste indentées
                c.drawString(x + 20, y, line)
            y -= 11

    y -= 6


# ========================
# PROJETS PERSONNELS
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Projets personnels")
y -= 20
c.setFont(police_2, size_text)

projects = [
    ("Assistant Clash Royal", [
        "but : donner les information de spectateur et créer un bot joueur",
        "Python, PyCharm, OpenCV, pyautogui"
    ]),
    ("Plugin Vinted (Android)", [
        "but : republier des annonces en un clic, générer une publication en fonction des photos",
        "Kotlin, Google MLkit, OpenCV, AccessibilityService"
    ]),
    ("Draft Analyzer (projet d'école)", [
        "Partie DPM : conception pour aider la communauté LoL et bookmakers",
        "Collecte de données (API + scraping) : compositions d'équipes, données sur champions et invocateurs",
        "Pipeline ML : XGBoost, fonction beta",
        # "Partie MLOps : pas encore commencée"
    ])
]

for title, details in projects:
    # Puce + titre du projet
    c.drawString(x - 5, y, bullet)
    c.setFont(police, size_text)
    c.drawString(x, y, title)
    text_width = c.stringWidth(title, police, size_text)
    # c.line(x, y - 1, x + text_width, y - 1)  # souligne le titre
    y -= 10

    # Détails du projet, retours à la ligne
    c.setFont(police_2, size_text)
    for d in details:
        for line in d.split("\n"):
            c.drawString(x + 10, y, line)
            y -= 10
    y -= 6

# # ========================
# # PROJETS PERSONNELS
# # ========================
# y -= 20
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Projets personnels")
# # c.drawString(x, y, "Projets personnels")
# y -= 20
# c.setFont(police_2, size_text)
# projects = [
#     "Assistant Clash Royal : Python, PyCharm, openCV, pyautogui",
#     "Plugin Vinted (Android) : Kotlin, Google MLkit, OpenCV, AccessibilityService"
# ]
# for p in projects:
#     c.drawString(x, y, p)
#     y -= 12

# # ========================
# # FORMATION
# # ========================
# y -= 40
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Formation")
# # c.drawString(x, y, "Formation")
# y -= 20
# c.setFont(police_2, size_text)

# formation = [
#     "Juin 2025 - Juin 2026 : Formation Machine Learning Engineer chez Datascientest (Axa)",
#     "2015-2018 : ENSSAT Lannion - diplôme d’ingénieur en informatique",
#     "2018 : Erasmus Universidad Politécnica, Madrid (métaheuristiques, biologie programmable, modèles de raisonnement)",
#     "2013-2015 : CPGE PCSI-PSI au Lycée Michelet de Vanves"
# ]

# for f in formation:
#     c.drawString(x, y, f)
#     y -= 14


# ========================
# FORMATION
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Formation")
y -= 20

date_color = blue

formation = [
    ("Juin 2025 - Juin 2026", "Datascientest / Axa", "Formation Machine Learning Engineer – alternance Data Science"),
    ("Sept 2015 - Juin 2018", "ENSSAT Lannion", "Diplôme d’ingénieur en informatique"),
    ("Sept 2017 - Fév 2018", "Universidad Politécnica, Madrid", "Erasmus – Métaheuristiques, biologie programmable, modèles de raisonnement"),
    ("Sept 2013 - Juin 2015", "Lycée Michelet de Vanves", "CPGE PCSI-PSI")
]

for date, school, detail in formation:
    y-=10
    # Date
    c.setFillColor(date_color)
    c.setFont(police_2 + "-Oblique", size_text)
    c.drawString(x, y, date)

    # École / établissement
    c.setFillColor(dark)
    c.setFont(police, size_text)
    c.drawString(x, y + 10, school)

    y -= 12

    # Détails / diplôme / spécialité
    c.setFont(police_2, size_text)
    c.drawString(x, y, detail)
    y -= 15



# ========================
# ENREGISTREMENT PDF
# ========================
c.save()

NameError: name 'company_width' is not defined

In [23]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.lib.colors import HexColor
from reportlab.graphics import renderPDF
from svglib.svglib import svg2rlg
from reportlab.lib.utils import simpleSplit


def draw_justified_text(c, text, x, y, max_width, font, font_size, line_height=12):
    c.setFont(font, font_size)

    lines = simpleSplit(text, font, font_size, max_width)

    for i, line in enumerate(lines):

        # dernière ligne → alignée à gauche
        if i == len(lines) - 1:
            c.drawString(x, y, line)

        else:
            words = line.split(" ")
            if len(words) == 1:
                c.drawString(x, y, line)
            else:
                total_words_width = sum(c.stringWidth(w, font, font_size) for w in words)
                space_width = c.stringWidth(" ", font, font_size)

                total_space = max_width - total_words_width
                space_between = total_space / (len(words) - 1)

                current_x = x

                for w in words:
                    c.drawString(current_x, y, w)
                    current_x += c.stringWidth(w, font, font_size) + space_between

        y -= line_height

    return y


def draw_colored_line(c, debut_x, y, longueur, couleur):
    c.saveState()
    c.translate(debut_x, y)
    c.setFillColor(couleur)
    c.rect(0, 0, longueur, 1, fill=1, stroke=0)
    c.restoreState()

def draw_bandeau(c, debut_x, debut_y, longueur, couleur, titre):
    c.saveState()
    c.translate(debut_x - 5, debut_y - 5)
    c.setFillColor(couleur)
    c.rect(0, 0, longueur, 20, fill=1, stroke=0)

    # c.translate(debut_x + longueur - 20, debut_y - 5)
    # c.translate(debut_x, debut_y - 5)
    c.translate(longueur - 23, -3)
    c.rotate(49)
    
    c.setFillColor(white)
    c.rect(0, 0, 30, 4, fill=1, stroke=0)

    c.translate(10,-10)
    c.rect(0, 0, 30, 4, fill=1, stroke=0)

    c.restoreState()
    c.setFont(police, size_big_text)
    c.drawString(x, y, f"{titre}")
    

# ========================
# INITIALISATION
# ========================
c = canvas.Canvas("cv_samuel_ramirez.pdf", pagesize=A4)
width, height = A4
police = "Helvetica-Bold"
police_2 = "Helvetica"
size_text = 9
size_big_text = 12

# Couleurs
blue = HexColor("#284E73")
light_blue = HexColor("#ADD1EA")
# couleur_colonne = HexColor("#317A34")
couleur_colonne = blue
light_grey = HexColor("#F2F2F2")
dark = HexColor("#333333")
white = HexColor("#FFFFFF")

# Largeur colonne gauche
left_w = 135
x = left_w/2
marge_colonne = 12
bullet = "•"

# ========================
# FOND COLONNE GAUCHE
# ========================
c.setFillColor(couleur_colonne)
c.rect(0, 0, left_w, height, fill=1, stroke=0)

# ========================
# PHOTO
# ========================
y = height
largeur_photo = 86
hauteur_photo = 115
photo_x = x - largeur_photo/2
y = y - (hauteur_photo + 20)
photo_y = y
c.drawImage(r"D:\CV2026\photo_cv.png", photo_x, photo_y, width=largeur_photo, height=hauteur_photo)

# ========================
# NOM ET DATE DE NAISSANCE
# ========================
c.setFillColor(white)
c.setFont(police, size_big_text)
y -= 25
c.drawCentredString(x, y, "Samuel RAMIREZ")
c.setFont(police_2, size_text)
y -= 15
c.drawCentredString(x, y, "5 Février 1995")

# Ligne séparatrice
c.saveState()
longueur_trait = left_w - 20
y -= 12
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15

# ========================
# CONTACT
# ========================
contact = [
    ("phone.svg", "+33 6 81 44 65 34", None),
    ("envelope.svg", "samuel.reinaldo.ramirez\n@gmail.com", None),
    ("home.svg", "Arcueil 94110, France", None),
    # ("car.svg", "Permis B", None),
    ("linkedin.svg", "LinkedIn", "https://www.linkedin.com/in/samuel-ramirez-53064718b/")
]


c.setFont(police_2, size_text)
icon_size = 12
line_height = 12

for svg_file, text, link in contact:
    lines = text.split("\n")
    n_lines = len(lines)

    # Charger et recolorer SVG
    drawing = svg2rlg(svg_file)
    def recolor(shape):
        if hasattr(shape, 'fillColor') and shape.fillColor is not None:
            shape.fillColor = white
        if hasattr(shape, 'strokeColor') and shape.strokeColor is not None:
            shape.strokeColor = white
        if hasattr(shape, 'contents'):
            for s in shape.contents:
                recolor(s)
    recolor(drawing)

    # Redimensionner
    scale = icon_size / max(drawing.width, drawing.height)
    drawing.width *= scale
    drawing.height *= scale

    # Dessiner l'icône alignée à la première ligne
    icon_y = y - (line_height * (n_lines - 1))/2 - icon_size/2 + 4
    renderPDF.draw(drawing, c, marge_colonne - 2, icon_y)
    

    # Dessiner texte
    for line in lines:
        c.drawString(marge_colonne + icon_size + 3, y, line)
        y -= line_height

    # longueur_hypertext = marge_colonne + icon_size + 40
    longueur_hypertext = left_w - marge_colonne
    debut_hypertext_x = marge_colonne - 2
    # Lien
    if link:
        c.linkURL(link, (debut_hypertext_x, y + 5, longueur_hypertext, y + line_height*n_lines+10), relative=0)
        # c.linkURL(link, (20, y-2, left_w-50, y + line_height*n_lines), relative=0)

    y -= 4

# ========================
# LANGUES
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Langues")
c.setFont(police_2, size_text)
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y-=15

# langs = [
#     "Français : langue maternelle",
#     "Anglais : 850 au TOEIC",
#     "Espagnol : langue maternelle"
# ]

langs = [
    "Français : C2",
    "Anglais : C1",
    "Espagnol : C2",
    "Hébreu : B1",
    # "Chinois : A1"
]

for l in langs:
    c.drawString(marge_colonne, y, l)
    y -= 12

# ========================
# CERTIFICATIONS
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Certifications")
c.setFont(police_2, size_text)
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y-=15

certs = [
    "Coursera : Machine Learning \nStanford Uni. par Andrew Ng",
    "OCA Java SE 8",
    "OCP Java SE 8",
    "AWS Certified Cloud \nPractitioner",
    "Angular Certification level 1"
]

for ccc in certs:
    c.drawString(marge_colonne - 5, y, bullet)
    for line in ccc.split("\n"):
        c.drawString(marge_colonne, y, line)
        y -= 8
    y -= 4



# ========================
# COMPETENCES
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16

c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Compétences")
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15
c.setFont(police_2, size_text)
# skills = [
#     "Versioning (Git), Agilité (Scrum)",
#     "Bases de données SQL, AWS, Xano",
#     "Matlab, POO, Programmation fonctionnelle",
#     "Métaheuristiques, ETL : Talend, Databricks, PySpark",
#     "Développement applicatif : ",
#     "Front : Javascript, HTML, CSS, Angular, React",
#     "Back : Java 8, Python, REST",
#     "Mobile : Android, Flutter"
# ]

# # Versioning (Git), Agilité (scrum), Algorithmie, Bases de données SQL, AWS, Xano,
# # Utilisation du terminal, Matlab, Programmation orientée objet, Programmation
# # fonctionnelle, Métaheuristiques, ETL : Talend, Databricks, PySpark
# # Développement applicatif :
# # Front : Javascript, HTML, css, Angular, React
# # Back : Java 8 et technologies associés, Python, REST
# # Mobile : Android, Flutter


# max_width = left_w - marge_colonne - 10

# for s in skills:
#     lines = simpleSplit(s, police_2, size_text, max_width)

#     for line in lines:
#         c.drawString(marge_colonne, y, line)
#         y -= 10

#     y -= 2

skills = [
    "Machine Learning\nSupervisé, non supervisé,\nfeature engineering",
    
    "Clustering & anomalies\nRéduction de dimension\nSéries temporelles",
    
    "Deep Learning\nKeras, CNN, OpenCV",
    
    "Data Engineering\nSQL, Talend, PySpark",
    
    "MLOps\nDocker, MLflow, DVC\nAirflow, BentoML",
    
    "Monitoring\nPrometheus, Grafana",
    
    "APIs & Serving\nFastAPI, REST, AWS",
    
    "Développement\nPython, Java, React, Git\nLinux / Bash, Android, Flutter",
    
    "Méthodes\nAgile (Scrum)"
]

for s in skills:

    c.drawString(marge_colonne - 5, y, bullet)

    first = True
    for line in s.split("\n"):

        if first:
            c.setFont(police, size_text)  # titre en gras
            c.drawString(marge_colonne, y, line)

            text_width = c.stringWidth(line, police, size_text)
            c.line(marge_colonne, y - 1, marge_colonne + text_width, y - 1)

            first = False
        else:
            c.setFont(police_2, size_text)
            c.drawString(marge_colonne, y, line)

        y -= 8

    y -= 4


# ========================
# CENTRES D'INTERET
# ========================
y -= 5
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 16
c.setFont(police, size_big_text)
c.drawCentredString(left_w/2, y, "Centres d'intérêt")
y -= 10
draw_colored_line(c, marge_colonne, y, longueur_trait, white)
y -= 15
c.setFont(police_2, size_text)
interets = [
    "Judo  " + bullet + "  Futsal  " + bullet + "  Cyclisme",
    "Échecs  " + bullet + "  E-Sport  " + bullet + "  Trading", 
    "Développement mobile  "
]
for i in interets:
    c.drawString(marge_colonne, y, i)
    y -= 12
# ========================
# COLONNE DROITE
# ========================
marge_colonne_droite = 10
x_title = (width + left_w) / 2
x = left_w + marge_colonne_droite
y = height - 30
c.setFillColor(dark)

# Titre
c.setFont(police, 18)
c.drawCentredString(x_title, y, "Ingénieur en Informatique - Data Scientist")

# ========================
# PROFIL
# ========================
y -= 40
c.setFont(police, size_big_text)
c.drawString(x, y, "Profil")
draw_bandeau(c, x, y, 440, light_blue, "Profil")
y -= 20

profil_text = (
"Ingénieur en informatique fasciné par l’intelligence artificielle et la "
"modélisation prédictive. Après cinq ans comme développeur web, j’ai exploré "
"l’entrepreneuriat en co-créant l’application mobile Yummap, avant de me "
"spécialiser en Data Science. Actuellement Data Scientist en alternance chez "
"AXA, je conçois des modèles data tout en développant des projets personnels "
"en machine learning, notamment un modèle de prédiction de l’issue d’une "
"partie de League of Legends à partir de la draft."
)

max_width = width - x - 40

y = draw_justified_text(
    c,
    profil_text,
    x,
    y,
    max_width,
    police_2,
    size_text,
    11
)


# # ========================
# # EXPERIENCES PROFESSIONNELLES
# # ========================
# y -= 20
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Expériences professionnelles")
# y -= 20

# date_color = blue

# exp = [

# ("Juin 2025 – Juin 2026", "AXA – Data Scientist (alternance)", [
# "Développement et optimisation d’un algorithme de name matching pour la détection de personnes \nsous sanction dans la base clients.",
# "Réduction du ratio de faux positifs de 4 alertes pour 1 réelle à 1,7 grâce à l’exploration de \ncombinaisons de distances de chaînes.",
# "Optimisation du temps de traitement de plusieurs heures à ~50 minutes via des méthodes \nde clustering (médoïdes).",
# "Étude d’un système de classification automatique des alertes liées aux \nPersonnes Politiquement Exposées (PPE)."
# ]),

# ("Fév 2024 – Jan 2025", "Yummap – Co-fondateur", [
# "Conception et développement d’une application mobile de découverte de restaurants.",
# "Développement Flutter et application d’administration (WeWeb puis Flutter).",
# "Architecture backend : Xano, stockage AWS S3.",
# "Marketing et acquisition : création de contenu, campagnes emailing.",
# "Prospection commerciale et rendez-vous clients."
# ]),

# ("Fév 2018 – Déc 2023", "Oxyl – Consultant développeur WEB", [
# "• Août 2022 - Oct 2023 : Développeur JS (React / Node), Arcueil\n"
# "Développement du projet Autocom pour automatiser le recrutement et la partie commerciale d’Oxyl. Scraping de sites, monitoring des bots, maintenance et évolution avec une équipe agile de 5 développeurs.",
# "• Oct 2021 - Juil 2022 : Développeur Java chez Lafarge Holcim, Clamart\n"
# "Maintenance et évolution de Quartz (ERP) pour la filière granulats. Nouvelle API pour stockage PDF, déploiements via GitLab.",
# "• Mai 2018 - Oct 2021 : Développeur web Capico, Arcueil\n"
# "Maintenance, évolution et tests unitaires du projet Java/Angular Capico. Suivi pédagogique et fonctionnel des cours et classes.",
# "• Sept 2019 - Juil 2022 : Développeur La Réunion Aérienne, Paris\n"
# "Maintenance et évolution des projets Java/JS NSI et Webreport en coopération avec la MOA.",
# "• Avr 2019 - Sept 2019 : Développeur / Scrum Master MNF (My Neuro Factory), Arcueil\n"
# "Application React/Java pour référencer les collaborateurs et faciliter la montée en compétences et certifications.",
# "• Déc 2018 - Avr 2019 : Développeur web consultant Wedia, Paris\n"
# "Développement d’une solution personnalisable de workflows (cycle en V), gestion des permissions selon type d’utilisateur."
# ])
# ]

# for date, company, bullets in exp:
#     y -= 10
#     # date
#     c.setFillColor(date_color)
#     c.setFont(police_2 + "-Oblique", size_text)
#     c.drawString(x, y, date)

#     # entreprise
#     c.setFillColor(dark)
#     c.setFont(police, size_text)
#     c.drawString(x, y + 10, company)

#     y -= 12

#     # bullets avec gestion du \n
#     c.setFont(police_2, size_text)
#     for b in bullets:
#         lines = b.split("\n")
#         for i, line in enumerate(lines):
#             if i == 0 and not line.startswith("Développement") and not line.startswith("Maintenance"):
#                 # première ligne → puce si ce n’est pas un retour à la ligne forcé
#                 c.drawString(x + 10, y, "• " + line)
#             else:
#                 # lignes suivantes → pas de puce, juste indentées
#                 c.drawString(x + 20, y, line)
#             y -= 11

#     y -= 6




# ========================
# EXPERIENCES PROFESSIONNELLES
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Expériences professionnelles")
y -= 20

date_color = blue

exp = [

("Juin 2025 – Juin 2026", "AXA – Data Scientist (alternance)", [
"Développement et optimisation d’un algorithme de name matching pour la détection de personnes \nsous sanction dans la base clients.",
"Réduction du ratio de faux positifs de 4 alertes pour 1 réelle à 1,7 grâce à l’exploration de \ncombinaisons de distances de chaînes.",
"Optimisation du temps de traitement de plusieurs heures à ~50 minutes via des méthodes \nde clustering (médoïdes).",
"Étude d’un système de classification automatique des alertes liées aux \nPersonnes Politiquement Exposées (PPE)."
]),

("Fév 2024 – Jan 2025", "Yummap – Co-fondateur", [
"Conception et développement d’une application mobile de découverte de restaurants.",
"Développement Flutter et application d’administration (WeWeb puis Flutter).",
"Architecture backend : Xano, stockage AWS S3.",
"Marketing et acquisition : création de contenu, campagnes emailing.",
"Prospection commerciale et rendez-vous clients."
]),

("Fév 2018 – Déc 2023", "Oxyl – Consultant développeur WEB", [
"Lafarge Holcim : maintenance et évolution de l’ERP Quartz (Java).",
"La Réunion Aérienne : évolution d’applications Java / JavaScript.",
"Wedia : développement d’une solution de workflows personnalisés (cycle en V).",
"Projets internes : My Neuro Factory (React, NodeJS, SQL, Scrum).",
"Développement d’outils internes (React, NodeJS, Puppeteer)."
]),

# ("00 - 00", "AA - ZZZ", [
# "Parler de Vinted, du travail à la ferme, des cours particuliers ?"
# ])

]

for date, company, bullets in exp:
    y -= 10
    # date
    c.setFillColor(date_color)
    c.setFont(police_2 + "-Oblique", size_text)
    c.drawString(x, y, date)

    # entreprise
    c.setFillColor(dark)
    c.setFont(police, size_text)
    c.drawString(x, y + 10, company)

    y -= 12

    # bullets avec gestion du \n
    c.setFont(police_2, size_text)
    for b in bullets:
        lines = b.split("\n")
        for i, line in enumerate(lines):
            if i == 0:
                # première ligne → puce
                c.drawString(x + 10, y, "• " + line)
            else:
                # lignes suivantes → pas de puce, juste indentées
                c.drawString(x + 20, y, line)
            y -= 11

    y -= 6


# ========================
# PROJETS PERSONNELS
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Projets personnels")
y -= 20
c.setFont(police_2, size_text)

projects = [
    ("Assistant Clash Royal", [
        "but : donner les information de spectateur et créer un bot joueur",
        "Python, PyCharm, OpenCV, pyautogui"
    ]),
    ("Plugin Vinted (Android)", [
        "but : republier des annonces en un clic, générer une publication en fonction des photos",
        "Kotlin, Google MLkit, OpenCV, AccessibilityService"
    ]),
    ("Draft Analyzer (projet d'école)", [
        "Partie DPM : conception pour aider la communauté LoL et bookmakers",
        "Collecte de données (API + scraping) : compositions d'équipes, données sur champions et invocateurs",
        "Pipeline ML : XGBoost, fonction beta",
        # "Partie MLOps : pas encore commencée"
    ])
]

for title, details in projects:
    # Puce + titre du projet
    c.drawString(x - 5, y, bullet)
    c.setFont(police, size_text)
    c.drawString(x, y, title)
    text_width = c.stringWidth(title, police, size_text)
    # c.line(x, y - 1, x + text_width, y - 1)  # souligne le titre
    y -= 10

    # Détails du projet, retours à la ligne
    c.setFont(police_2, size_text)
    for d in details:
        for line in d.split("\n"):
            c.drawString(x + 10, y, line)
            y -= 10
    y -= 6

# # ========================
# # PROJETS PERSONNELS
# # ========================
# y -= 20
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Projets personnels")
# # c.drawString(x, y, "Projets personnels")
# y -= 20
# c.setFont(police_2, size_text)
# projects = [
#     "Assistant Clash Royal : Python, PyCharm, openCV, pyautogui",
#     "Plugin Vinted (Android) : Kotlin, Google MLkit, OpenCV, AccessibilityService"
# ]
# for p in projects:
#     c.drawString(x, y, p)
#     y -= 12

# # ========================
# # FORMATION
# # ========================
# y -= 40
# c.setFont(police, size_big_text)
# draw_bandeau(c, x, y, 440, light_blue, "Formation")
# # c.drawString(x, y, "Formation")
# y -= 20
# c.setFont(police_2, size_text)

# formation = [
#     "Juin 2025 - Juin 2026 : Formation Machine Learning Engineer chez Datascientest (Axa)",
#     "2015-2018 : ENSSAT Lannion - diplôme d’ingénieur en informatique",
#     "2018 : Erasmus Universidad Politécnica, Madrid (métaheuristiques, biologie programmable, modèles de raisonnement)",
#     "2013-2015 : CPGE PCSI-PSI au Lycée Michelet de Vanves"
# ]

# for f in formation:
#     c.drawString(x, y, f)
#     y -= 14


# ========================
# FORMATION
# ========================
y -= 20
c.setFont(police, size_big_text)
draw_bandeau(c, x, y, 440, light_blue, "Formation")
y -= 20

date_color = blue

formation = [
    ("Juin 2025 - Juin 2026", "Datascientest / Axa", "Formation Machine Learning Engineer – alternance Data Science"),
    ("Sept 2015 - Juin 2018", "ENSSAT Lannion", "Diplôme d’ingénieur en informatique"),
    ("Sept 2017 - Fév 2018", "Universidad Politécnica, Madrid", "Erasmus – Métaheuristiques, biologie programmable, modèles de raisonnement"),
    ("Sept 2013 - Juin 2015", "Lycée Michelet de Vanves", "CPGE PCSI-PSI")
]

for date, school, detail in formation:
    y-=10

    # École / établissement
    c.setFillColor(dark)
    c.setFont(police, size_text)
    c.drawString(x, y + 10, school)
    # school_width = c.stringWidth(school, police, size_text)

    # Date
    c.setFillColor(date_color)
    c.setFont(police_2 + "-Oblique", size_text)
    # c.drawString(x + school_width + 10, y + 10, date)
    c.drawString(x, y, date)

    y -= 12

    # Détails / diplôme / spécialité
    c.setFillColor(dark)
    c.setFont(police_2, size_text)
    c.drawString(x, y, detail)
    y -= 15



# ========================
# ENREGISTREMENT PDF
# ========================
c.save()